In [1]:
import torch
import json
import random
import re
import ast
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

In [22]:
# load the sentences and correct labels
all_sentences = []

with open("../01_data/annotations.json", "r") as f:
    data = json.load(f)

for sentence in data:
    text = sentence["data"]["sentence"]
    labels = []
    results = sentence["annotations"][0]["result"]
    labels = [r["value"]["text"] for r in results]
    
    all_sentences.append({
        "text": text,
        "labels": labels
    })

In [32]:
def create_llm_annotations(text, spans):

    # sort the spans according to their start and end index
    spans = sorted(spans, key=lambda x: x["start"])

    # store the text and set index variable
    llm_text = ""
    last_idx = 0

    # loop through spans and add the span with custom characters
    for span in spans:
        llm_text += text[last_idx:span["start"]]
        llm_text += f"@@{text[span["start"]:span["end"]]}##"
        last_idx = span["end"]

    # add the rest of the text
    llm_text += text[last_idx:]

    return llm_text

def llm_output_to_bio(annotated_text):

    # split words via a regex
    words = re.findall(r"@@.*?##|[\w'-]+|[^\w\s]", annotated_text)

    # empty list to store the bio tags
    bio_tags = []

    # loop through all words
    for word in words:

        # if it is an annotated span, split words and assign bio labels
        if word.startswith("@@") and word.endswith("##"):
            entity_text = word[2:-2]
            entity_words = re.findall(r"[\w'-]+|[^\w\s]", entity_text)
            for i, t in enumerate(entity_words):
                tag = "B-sg" if i == 0 else "I-sg"
                bio_tags.append((t, tag))
        else:
            # otherwise assign O tag
            bio_tags.append((word, "O"))

    return bio_tags

# initialize empty dataset list
dataset = []

# loop through all sentences in the data
for task in data:
    # get the sentence and all annotations
    text = task["data"]["sentence"]
    results = task["annotations"][0]["result"]
    labels = [r["value"]["text"] for r in results]

    # store annotation spans in a list
    spans = [
        {
            "start": r["value"]["start"],
            "end": r["value"]["end"],
            "labels": r["value"]["labels"][0][0:2]
        }
        for r in results if r["type"] == "labels"
    ]
   
    llm_text = create_llm_annotations(text, spans)
    bio_tags = llm_output_to_bio(llm_text)

    # add everything to the dataset list
    dataset.append({
        "text": text,
        "labels": labels,
        "llm_text": llm_text,
        "bio_tags": bio_tags
    })

In [34]:
# load the model
checkpoint = "HuggingFaceTB/SmolLM-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

In [52]:
# define chat template for normal inference
def compile_ner_prompt(few_shot_examples, test_sentence):
    chat = [
          {"role": "system",
           "content": (
                "You are a helpful assistant that extracts social group mentions from text.\n"
                "Definition of a social group: A social group is a segment of society or a collection of people who share common socio-demographic traits or attributes that are ascriptive and/or acquired. \n"
                "These include characteristics like sex and gender, age, ethnicity, language, religion, place of residence, nationality, income, occupation, education and more. \n"
                "Implicit social group references such as people, everyone, communities, the public, or the nation are excluded. \n"
                "This definition excludes institutionally organized groups and state authorities like interest groups, trade unions, the police, and business entities. \n"
                "Groupings of individuals within institutionally organized groups and state authorities are included as long as the defining feature of the group is a common socio-demographic trait or attribute (e.g. workers, union members, police officers, business owners, teachers). \n"
                "Groupings based on shared beliefs, life experiences, ideology, party affiliation and/or political opinion are excluded.\n\n"
                "Your task is to mark all social group mentions from a given sentence.\n"
                "\n Respond with the entire sentence and mark the start of the group mention with @@ and the end with ##."
                "If there are no social group mentions in the sentence, respond with: None."
                )
        }
    ]

    # add few-shot examples
    for example in few_shot_examples:
        context = example["text"]
        if example["labels"]:
            answer = example["llm_text"]
        else:
            answer = "None"
        chat.append(
            {"role": "user", "content": f"Sentence: {context}"})
        chat.append({"role": "assistant", "content": answer})
    
    # add the test sentence
    chat.append(
        {"role": "user", "content": f"Sentence: {test_sentence}"})

    # compile the prompt
    prompt = tokenizer.apply_chat_template(
    chat, return_tensors="pt", tokenize=False, add_generation_prompt=True)
    return prompt

In [83]:
# create some few-shot examples
non_empty_examples = [ex for ex in dataset if ex["labels"]]
empty_examples = [ex for ex in dataset if not ex["labels"]]
few_shot_examples = random.sample(non_empty_examples, 4) + random.sample(empty_examples, 1)

# create test dataset
split_idx = int(len(non_empty_examples)*0.95)
test_dataset = non_empty_examples[split_idx:] + random.sample(empty_examples, int(len(empty_examples)*0.01))
random.shuffle(test_dataset)

In [85]:
compile_ner_prompt(few_shot_examples, "Hey there!")

"<|im_start|>system\nYou are a helpful assistant that extracts social group mentions from text.\nDefinition of a social group: A social group is a segment of society or a collection of people who share common socio-demographic traits or attributes that are ascriptive and/or acquired. \nThese include characteristics like sex and gender, age, ethnicity, language, religion, place of residence, nationality, income, occupation, education and more. \nImplicit social group references such as people, everyone, communities, the public, or the nation are excluded. \nThis definition excludes institutionally organized groups and state authorities like interest groups, trade unions, the police, and business entities. \nGroupings of individuals within institutionally organized groups and state authorities are included as long as the defining feature of the group is a common socio-demographic trait or attribute (e.g. workers, union members, police officers, business owners, teachers). \nGroupings bas

In [86]:
# generate the answers for the normal format and store in a list
gen_answers = []

for i in range(len(test_dataset)):
    sentence = test_dataset[i]["text"]
    prompt = compile_ner_prompt(few_shot_examples, sentence)
    prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = model.generate(**prompt_ids)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens = True)
    answer_text = generated_text.split("assistant\n")[-1]
    
    # convert the generated prediction to the bio scheme
    answer_bio = llm_output_to_bio(answer_text)
    #answer_list = to_list_or_empty(answer)
    gen_answers.append({"text": answer_text,
                        "bio": answer_bio})

In [87]:
# show a few predictions
for idx in range(10):
    input = test_dataset[idx]["text"]
    prediction = gen_answers[idx]["text"]
    print(f"Input: {input}")
    print(f"prediction: {prediction}")
    print("-"*100)

Input: As the hon. Gentleman knows and, I think, supported at the time, we have had to reduce the number of bases to ensure that our servicemen and women are in better accommodation in fewer remote areas, and in places where their spouses and partners have more chance of getting into employment.
prediction: As the hon. Gentleman knows and, I think, supported at the time, we have had to reduce the number of bases to ensure that our servicemen and women are in better accommodation in fewer
----------------------------------------------------------------------------------------------------
Input: Currently, 26% of the people sitting on FTSE 100 boards are women-more than ever before.
prediction: Currently, 26% of the people sitting on FTSE 100 boards are women, more than ever before.
----------------------------------------------------------------------------------------------------
Input: Will my right hon. Friend join me in congratulating the 150,000 young people who participate in cub 

In [99]:
# evaluate the generated answers

# evaluation at the word level

# get list of bio tags only
ground_truth_bio = [[tag for (_, tag) in sent["bio_tags"]] for sent in test_dataset]
pred_bio = [[tag for (_, tag) in sent["bio"]] for sent in gen_answers]

filtered_gt = []
filtered_pred = []
for gt, pred in zip(ground_truth_bio, pred_bio):
    if len(gt) == len(pred):
        filtered_gt.append(gt)
        filtered_pred.append(pred)

y_true = [tag for sent in filtered_gt for tag in sent]
y_pred = [tag for sent in filtered_pred for tag in sent]

# print the classification report
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

        B-sg       0.00      0.00      0.00        15
        I-sg       0.00      0.00      0.00        22
           O       0.92      1.00      0.96       407

    accuracy                           0.92       444
   macro avg       0.31      0.33      0.32       444
weighted avg       0.84      0.92      0.88       444



/Users/maxweiland/Desktop/SEDS/Master_Thesis/venv_thesis/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxweiland/Desktop/SEDS/Master_Thesis/venv_thesis/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/maxweiland/Desktop/SEDS/Master_Thesis/venv_thesis/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to

In [ ]:
# evaluate at the entity level

In [16]:
# evaluate the generated answers
results = []

for idx in range(len(test_dataset)):
    ground_truth = test_dataset[idx]["labels"]
    prediction = gen_answers[idx]
    results.append({"labels": ground_truth,
                    "prediction": prediction})

def evaluate_predictions(results):
    y_true = []
    y_pred = []

    for example in results:
        gold_mentions = set([m.lower().strip() for m in example["labels"]])
        pred_mentions = set([m.lower().strip() for m in example["prediction"]])

        for mention in gold_mentions:
            y_true.append(1)
            y_pred.append(1 if mention in pred_mentions else 0)

        for mention in pred_mentions:
            if mention not in gold_mentions:
                y_true.append(0)
                y_pred.append(1)

    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    return precision, recall, f1

p, r, f1 = evaluate_predictions(results)

print(f"Precision: {p:.4f} \nRecall: {r:.4f} \nF1: {f1:.4f}")

Precision: 0.1364 
Recall: 0.1667 
F1: 0.1500


In [17]:
# print some examples
for idx in range(5):
    print(test_dataset[idx]["text"])
    print(results[idx]["labels"])
    print(results[idx]["prediction"])
    print("-"*80)

"Such measures could include raising awareness of examples where local areas are taking a more informal approach to issues through, for example, restorative justice or working with potential offenders."
['potential offenders']
['restorative justice']
--------------------------------------------------------------------------------
It will protect smart meter services for both consumers and businesses by providing the enabling framework for a special administration regime for the national data and communications provider.
['consumers']
['smart meter services']
--------------------------------------------------------------------------------
I understand that there are currently 150 apprentices working on the site.
['apprentices']
['apprentices']
--------------------------------------------------------------------------------
"Friend the Member for Calder Valley (Craig Whittaker) referred-I visited him in Mytholmroyd to see some of the progress on them-and which were published last year, i